# TC Track Analogues – Workflow Notebook

This notebook walks through a complete attribution workflow for a single Tropical Cyclone (TC) case using the `tc_track_analogues` package.

**Steps covered:**
1. Load (or simulate) TC track ensembles for the factual and counterfactual climates.
2. Define the observed (target) track.
3. Find the best analogues in each ensemble.
4. Compute the probability ratio (attribution metric).
5. Visualise the results.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tc_track_analogues as tca

# Reproducible random number generator for the demo ensemble
rng = np.random.default_rng(42)

## 1. Define the target (observed) track

Replace the synthetic track below with the actual observed track loaded from your data source (e.g. IBTrACS, a CSV file, or an xarray Dataset).

In [ ]:
# Synthetic target track: westward-moving TC in the North Atlantic
n_steps = 40
target_track = pd.DataFrame({
    "lon": np.linspace(-60, -90, n_steps),
    "lat": np.linspace(15, 30, n_steps),
})

print(target_track.head())

## 2. Load ensemble tracks

Here we generate synthetic ensembles for demonstration purposes.
In a real application you would load these from files, e.g.:

```python
import xarray as xr

ds = xr.open_dataset('path/to/ensemble.nc')
factual_ensemble = [
    pd.DataFrame({'lon': ds.lon.sel(member=i).values,
                  'lat': ds.lat.sel(member=i).values})
    for i in ds.member
]
```

In [ ]:
def make_ensemble(n_members: int, base_track: pd.DataFrame, noise_scale: float) -> list:
    """Create a synthetic ensemble by adding Gaussian noise to a base track."""
    return [
        pd.DataFrame({
            "lon": base_track["lon"].values + rng.normal(0, noise_scale, len(base_track)),
            "lat": base_track["lat"].values + rng.normal(0, noise_scale, len(base_track)),
        })
        for _ in range(n_members)
    ]

# Factual ensemble: tracks similar to the observed one
factual_ensemble = make_ensemble(200, target_track, noise_scale=2.0)

# Counterfactual ensemble: systematically shifted (simulating a climate-change effect)
shifted_base = target_track.copy()
shifted_base["lat"] = shifted_base["lat"] - 3  # shifted equatorward in counterfactual
counterfactual_ensemble = make_ensemble(200, shifted_base, noise_scale=2.0)

print(f"Factual ensemble size:         {len(factual_ensemble)} members")
print(f"Counterfactual ensemble size:  {len(counterfactual_ensemble)} members")

## 3. Find analogues

In [ ]:
N_ANALOGUES = 20

factual_analogues = tca.find_analogues(
    target_track, factual_ensemble, n_analogues=N_ANALOGUES
)
counterfactual_analogues = tca.find_analogues(
    target_track, counterfactual_ensemble, n_analogues=N_ANALOGUES
)

print("Closest factual analogues:")
print(factual_analogues.head(5))
print("\nClosest counterfactual analogues:")
print(counterfactual_analogues.head(5))

## 4. Compute attribution metric

In [ ]:
result = tca.compute_attribution(
    factual_ensemble,
    counterfactual_ensemble,
    target_track,
    n_analogues=N_ANALOGUES,
)

print(f"Probability ratio (factual / counterfactual): {result['probability_ratio']:.3f}")
print(f"Mean distance – factual:                      {result['mean_distance_factual']:.3f}")
print(f"Mean distance – counterfactual:               {result['mean_distance_counterfactual']:.3f}")

## 5. Visualise the analogues

In [ ]:
fig = tca.plot.plot_analogues(
    target_track,
    result["factual_analogues"],
    factual_ensemble,
    result["counterfactual_analogues"],
    counterfactual_ensemble,
)
plt.show()

## 6. Distance distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(
    result["factual_analogues"]["distance"],
    bins=20, alpha=0.6, label="Factual", color="steelblue",
)
ax.hist(
    result["counterfactual_analogues"]["distance"],
    bins=20, alpha=0.6, label="Counterfactual", color="darkorange",
)
ax.set_xlabel("Track distance (degrees)")
ax.set_ylabel("Count")
ax.set_title("Distribution of analogue distances")
ax.legend()
plt.tight_layout()
plt.show()